# Capstone 2 — Tic-Tac-Toe with an optimal opponent

Two players share a 3 by 3 board. The human plays `X`. The computer plays `O` and chooses with minimax, so it never loses if both sides play perfectly. A draw is the best available result against this opponent.

## Representation

The board is a tuple of nine cells. A tuple is immutable, so each imagined move in the search returns a new board instead of mutating the live one. Cells are `"X"`, `"O"`, or `" "`.

## Minimax

For a position, the score is `+1` if `O` has won, `-1` if `X` has won, and `0` for a draw. `O` chooses the move with the maximum score. `X` is modelled as choosing the move with the minimum score. The search is exhaustive: there are at most `9!` leaves, and far fewer once wins are cut off. No pruning is required for a board this small.


In [1]:
from dataclasses import dataclass

LINES = (
    (0, 1, 2), (3, 4, 5), (6, 7, 8),
    (0, 3, 6), (1, 4, 7), (2, 5, 8),
    (0, 4, 8), (2, 4, 6),
)
EMPTY = " "


def winner(board: tuple[str, ...]) -> str | None:
    for a, b, c in LINES:
        if board[a] != EMPTY and board[a] == board[b] == board[c]:
            return board[a]
    return None


def full(board: tuple[str, ...]) -> bool:
    return EMPTY not in board


def children(board: tuple[str, ...], mark: str):
    for index, cell in enumerate(board):
        if cell == EMPTY:
            yield index, board[:index] + (mark,) + board[index + 1:]


def minimax(board: tuple[str, ...], mark: str) -> int:
    found = winner(board)
    if found == "O":
        return 1
    if found == "X":
        return -1
    if full(board):
        return 0
    scores = [minimax(next_board, "X" if mark == "O" else "O") for _, next_board in children(board, mark)]
    return max(scores) if mark == "O" else min(scores)


def best_move(board: tuple[str, ...]) -> int:
    ranked = []
    for index, next_board in children(board, "O"):
        ranked.append((minimax(next_board, "X"), index))
    return max(ranked)[1]


def render(board: tuple[str, ...]) -> str:
    rows = [" | ".join(board[i:i + 3]) for i in range(0, 9, 3)]
    return "\n---------\n".join(rows)


@dataclass
class Game:
    board: tuple[str, ...] = (EMPTY,) * 9

    def play(self, index: int) -> str:
        if not 0 <= index <= 8 or self.board[index] != EMPTY:
            raise ValueError("illegal move")
        self.board = self.board[:index] + ("X",) + self.board[index + 1:]
        if winner(self.board) or full(self.board):
            return self.outcome()
        choice = best_move(self.board)
        self.board = self.board[:choice] + ("O",) + self.board[choice + 1:]
        return self.outcome()

    def outcome(self) -> str:
        found = winner(self.board)
        if found:
            return f"{found} wins"
        if full(self.board):
            return "draw"
        return "in progress"


game = Game()
preferred = [0, 2, 6, 5, 7, 3, 1, 8]
for move in preferred:
    if game.outcome() != "in progress" or game.board[move] != EMPTY:
        continue
    status = game.play(move)
    print(f"human plays {move} -> {status}")
    print(render(game.board))
    print()
print("final:", game.outcome())


human plays 0 -> in progress
X |   |  
---------
  | O |  
---------
  |   |  

human plays 2 -> in progress
X | O | X
---------
  | O |  
---------
  |   |  

human plays 6 -> O wins
X | O | X
---------
  | O |  
---------
X | O |  

final: O wins


## What to notice

The human opening in the corner does not produce a win. The engine takes the centre and answers every later threat. That is the observable consequence of minimax: the chosen move is the one whose worst reply is still best for the computer.

The same `play` method is the whole interface a notebook widget or a terminal loop would call. The scripted moves above are only the demonstration.
